# Qui va partir ? Prédiction du churn Revolut
**Binôme :** [Nom 1] — [Nom 2]  
**Track A** — ML for Business (MSA-DATB07-08)

**Recommandation (1 phrase) :** _à écrire à la fin_

In [1]:
# 0. Vérification des fichiers : le notebook doit tourner depuis ce dossier
from pathlib import Path

FILES = ['rev-users.csv', 'rev-transactions.csv', 'rev-notifications.csv']
missing = [f for f in FILES if not Path(f).exists()]
assert not missing, f'Fichiers manquants : {missing}'
print('OK : tous les fichiers sont présents')

OK : tous les fichiers sont présents


## 1. Cadrage (avant tout modèle)

| Question | Réponse |
|---|---|
| Qui décide | l'équipe rétention |
| Unité | un client **actif** à la date de coupure (au moins une transaction dans les 60 jours avant) |
| Moment de la décision | à la date de coupure, une fois par mois |
| Ce qu'on fait de la sortie | on classe les clients par risque de départ `p` et on envoie l'offre aux premiers de la liste |
| Contrainte | budget de `BUDGET` offres par mois (hypothèse) |
| Fausse alerte | on paie l'offre `X` pour un client qui serait resté |
| Client raté | on perd sa valeur `Y` |

**Règle de décision.** Envoyer l'offre à un client rapporte en moyenne `p × Z × Y − X` :
chance qu'il parte × chance que l'offre le retienne × ce qu'il rapporte − prix de l'offre.
On envoie si ce gain est positif, c'est-à-dire si **p > X / (Z × Y)**.

| Chiffre | Valeur | Origine |
|---|---|---|
| `Y` valeur d'un client retenu (12 mois) | calculée ci-dessous | **données** (frais + commission carte) |
| `X` coût de l'offre | 5 € | **hypothèse** : un mois de Premium offert, coût réel pour Revolut inférieur au prix affiché |
| `Z` taux de réussite de l'offre | inconnu | **absent des données** (aucune offre envoyée) → on calcule le Z de point mort |
| `BUDGET` | 500 offres/mois | **hypothèse** à confirmer avec l'équipe |

## 2. Définition du churn

**Règle :** un client actif (au moins une transaction dans les 60 jours avant la coupure) est un churner s'il n'a **aucune transaction dans les 60 jours après**.

**Pourquoi 60 jours :** plus la fenêtre est courte, plus on étiquette comme partis des clients simplement en pause (cellule de vérification plus bas) :
- à 45 jours, 31 % des churners reviennent dans le mois suivant ;
- à 60 jours, 26 % ;
- 90 jours serait plus propre, mais repousserait toutes les coupures avant l'inscription d'une partie des clients.

→ Nous gardons 60 jours ; le taux de faux churn est une **limite assumée**.

**Fin des données :** les transactions s'arrêtent le 16 mai 2019, donc la fenêtre du label ne peut pas dépasser cette date.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

users = pd.read_csv('rev-users.csv', parse_dates=['created_date'])
notifs = pd.read_csv('rev-notifications.csv', parse_dates=['created_date'])
tx = pd.read_csv('rev-transactions.csv', parse_dates=['created_date'])
print(users.shape, notifs.shape, tx.shape)
print('Transactions :', tx.created_date.min(), '->', tx.created_date.max())

(19430, 12) (121813, 5) (2740075, 12)
Transactions : 2018-01-01 08:51:10.248709 -> 2019-05-16 18:22:16.959896


### 2.1 Vérification de la fenêtre : combien de « churners » reviennent ?

In [3]:
cutoff = pd.Timestamp('2019-01-15')

for w in [30, 45, 60, 90]:
    W = pd.Timedelta(days=w)
    before = tx[(tx.created_date >= cutoff - W) & (tx.created_date < cutoff)]
    after = tx[(tx.created_date >= cutoff) & (tx.created_date < cutoff + W)]
    actifs = set(before.user_id)
    churners = actifs - set(after.user_id)
    # reviennent dans les 30 jours qui suivent la fin de la fenêtre
    retour = tx[(tx.created_date >= cutoff + W) & (tx.created_date < cutoff + W + pd.Timedelta(days=30))]
    reviennent = churners & set(retour.user_id)
    print(f'{w} j : churn {len(churners)/len(actifs):.1%}, '
          f'reviennent dans le mois {len(reviennent)/len(churners):.1%}')

30 j : churn 17.2%, reviennent dans le mois 39.2%
45 j : churn 17.0%, reviennent dans le mois 31.0%


60 j : churn 16.2%, reviennent dans le mois 25.8%


90 j : churn 14.9%, reviennent dans le mois 15.9%


### 2.2 Dates de coupure (découpage dans le temps)

| Partie | Coupure | Fenêtre du label |
|---|---|---|
| Fit | 14/11/2018 | → 12/01/2019 |
| Validation | 14/01/2019 | → 14/03/2019 |
| Test (ouvert une seule fois) | 15/03/2019 | → 13/05/2019 |

**Règle : au moins 60 jours entre deux coupures.** Le label d'une partie doit être entièrement connu avant la coupure suivante. Sinon, le modèle a déjà vu une partie des réponses de la partie suivante : ce sont les mêmes clients, avec des features presque identiques. C'est une fuite (repérée par Héloïse avec des coupures espacées de 30 jours).

**Compromis :** l'entraînement date de novembre 2018. Si le comportement des clients a changé depuis (saisonnalité, dérive), le modèle est un peu en retard : à écrire dans les limites.

In [4]:
WINDOW = pd.Timedelta(days=60)
CUTOFFS = {
    'fit': pd.Timestamp('2018-11-14'),
    'validation': pd.Timestamp('2019-01-14'),
    'test': pd.Timestamp('2019-03-15'),
}
last_tx = tx.created_date.max()

for name, c in CUTOFFS.items():
    assert c + WINDOW <= last_tx, f'{name} : la fenêtre du label dépasse la fin des données'
    print(f'{name:<11} coupure {c.date()}  label jusqu\'au {(c + WINDOW).date()}')

# Nouveaux clients : ancienneté minimale à chaque coupure
last_signup = users.created_date.max()
for name, c in CUTOFFS.items():
    print(f'{name:<11} dernier inscrit il y a {(c - last_signup).days} jours')

# Pas de chevauchement : le label de chaque partie se termine avant la coupure suivante
dates = list(CUTOFFS.values())
for avant, apres in zip(dates, dates[1:]):
    assert avant + WINDOW <= apres, f'chevauchement entre {avant.date()} et {apres.date()}'
print('Aucun chevauchement entre les labels')

fit         coupure 2018-11-14  label jusqu'au 2019-01-13
validation  coupure 2019-01-14  label jusqu'au 2019-03-15
test        coupure 2019-03-15  label jusqu'au 2019-05-14
fit         dernier inscrit il y a -51 jours
validation  dernier inscrit il y a 10 jours
test        dernier inscrit il y a 70 jours
Aucun chevauchement entre les labels


### 2.3 Que faire des nouveaux clients ?

Aucune inscription après le 3 janvier 2019 : la part de clients récents baisse d'une coupure à l'autre, jusqu'à zéro au test. Or ils partent presque deux fois plus que les autres. Si on les garde, le modèle apprend sur une population et est évalué sur une autre.

**Décision : on exclut les clients inscrits depuis moins de 60 jours à la coupure, dans les trois parties.**
- Les trois parties comparent la même population.
- Il faut 60 jours d'historique pour calculer des features fiables.
- Un nouveau client relève de l'onboarding, pas de la rétention.

**Limite :** le modèle ne s'applique pas aux clients de moins de deux mois.

In [5]:
MIN_TENURE = pd.Timedelta(days=60)
signup = users.set_index('user_id').created_date

rows = []
for name, c in CUTOFFS.items():
    actifs = tx[(tx.created_date >= c - WINDOW) & (tx.created_date < c)].user_id.unique()
    revenus = set(tx[(tx.created_date >= c) & (tx.created_date < c + WINDOW)].user_id)
    d = pd.DataFrame({'user_id': actifs})
    d['nouveau'] = (c - d.user_id.map(signup)) < MIN_TENURE
    d['churn'] = ~d.user_id.isin(revenus)
    rows.append({'partie': name, 'actifs': len(d),
                 'nouveaux (<60 j)': int(d.nouveau.sum()),
                 'part nouveaux': f'{d.nouveau.mean():.1%}',
                 'churn nouveaux': f'{d[d.nouveau].churn.mean():.1%}' if d.nouveau.any() else '-',
                 'churn anciens': f'{d[~d.nouveau].churn.mean():.1%}'})
pd.DataFrame(rows)

,partie,actifs,nouveaux (<60 j),part nouveaux,churn nouveaux,churn anciens
0,fit,11217,2976,26.5%,22.8%,16.2%
1,validation,13658,3146,23.0%,23.8%,14.1%
2,test,13068,0,0.0%,-,13.0%


### 2.4 Valeur d'un client retenu `Y` (calculée sur les données)

Revenu observable par client actif et par mois, sur janvier–avril 2019, transactions terminées :
- les frais facturés (`FEE`) ;
- une commission sur les paiements carte : **0,3 %** (hypothèse, l'interchange européen est plafonné à 0,2 % en débit et 0,3 % en crédit).

Ce revenu sous-estime la valeur réelle (abonnements, change, etc. non visibles) : `Y` est donc **prudent**.

In [6]:
INTERCHANGE = 0.003   # hypothèse : commission sur paiement carte
USD_TO_EUR = 0.89     # hypothèse : taux moyen début 2019

periode = tx[(tx.created_date >= '2019-01-01') & (tx.created_date < '2019-05-01')
             & (tx.transactions_state == 'COMPLETED')]
n_clients = periode.user_id.nunique()
n_mois = 4

frais = periode.loc[periode.transactions_type == 'FEE', 'amount_usd'].sum()
carte = periode.loc[periode.transactions_type == 'CARD_PAYMENT', 'amount_usd'].sum()

revenu_mois_usd = (frais + INTERCHANGE * carte) / n_clients / n_mois
Y = revenu_mois_usd * 12 * USD_TO_EUR
print(f'Revenu par client actif : {revenu_mois_usd:.2f} $/mois  ->  Y = {Y:.1f} € sur 12 mois')

Revenu par client actif : 1.58 $/mois  ->  Y = 16.8 € sur 12 mois


### 2.5 Coût de l'offre, point mort et sensibilité

`Z` n'est pas dans les données. Au lieu de le deviner, on calcule le **Z minimum** pour que l'offre soit rentable, même sur un client sûr de partir (`p = 1`) : `Z > X / Y`.

In [7]:
X = 5          # hypothèse : coût d'une offre en €
BUDGET = 500   # hypothèse : offres par mois

Z_point_mort = X / Y
print(f'Z de point mort : {Z_point_mort:.0%}  (en dessous, aucun modèle ne rend l\'offre rentable)')

# Sensibilité : seuil de risque p au-dessus duquel on envoie l'offre
rows = []
for x in [2, 5, 8]:
    for z in [0.25, 0.35, 0.50]:
        seuil = x / (z * Y)
        rows.append({'X (€)': x, 'Z': f'{z:.0%}',
                     'seuil p': round(seuil, 2) if seuil <= 1 else 'jamais rentable'})
pd.DataFrame(rows).pivot(index='X (€)', columns='Z', values='seuil p')

Z de point mort : 30%  (en dessous, aucun modèle ne rend l'offre rentable)


Z,25%,35%,50%
X (€),,,
2,0.48,0.34,0.24
5,jamais rentable,0.85,0.59
8,jamais rentable,jamais rentable,0.95


## 3. Audit des données
_Lignes impossibles, colonnes constantes, valeurs manquantes, clients sans transaction._

## 4. Découpage dans le temps : fit / validation / test
| Partie | Coupure | Fenêtre du label |
|---|---|---|
| Fit | [date] | [date] → [date] |
| Validation | [date] | [date] → [date] |
| Test (ouvert une fois) | [date] | [date] → [date] |

## 5. Features + test anti-fuite
| Colonne | Connue au moment de décider ? | Décision |
|---|---|---|
| `plan` | non : photo prise à l'export | retirée |
| `num_referrals`, `num_successful_referrals` | toujours 0 | retirées |
| ... | ... | ... |

## 6. Baseline vs modèles
- Baseline : _règle en une ligne_
- Régression logistique
- Modèle à base d'arbres

Métriques : PR-AUC (comparée au taux de churn) + € pour 1000 clients. Pas d'accuracy.

## 7. Seuil fixé par les coûts et budget (choisi sur la validation)

## 8. Test final — **ouvert une seule fois ici**

## 9. Où le modèle se trompe
_Tableau ou graphe : pires erreurs, groupes faibles (pays, ancienneté, activité)._

## 10. Limites et recommandation
- Ce que les données ne contiennent pas : _..._
- Hypothèses : _..._
- **Décision :** on lance / on ne lance pas / on lance si _..._
- À surveiller le premier mois : _..._